# Training on Colab — task #38

Two-stage transfer learning for EfficientNet-B0 and ResNet50 on Grocery Store Dataset.
Runs the same code as locally — this notebook is just the Colab entry point.

**Before running:** Runtime → Change runtime type → GPU (T4).

## 1. Clone repo and dataset

In [ ]:
import os

if not os.path.exists('/content/hotdog'):
    !git clone --branch ml https://github.com/dsusachev/hotdog.git /content/hotdog
if not os.path.exists('/content/hotdog/ml/dataset/GroceryStoreDataset'):
    !git clone https://github.com/marcusklasson/GroceryStoreDataset.git /content/hotdog/ml/dataset/GroceryStoreDataset

%cd /content/hotdog

## 2. Install dependencies
Colab ships with torch/torchvision/pillow/sklearn — only what's missing is installed.

In [ ]:
!pip install -q -r ml/requirements.txt

## 3. Sanity-check that the pipeline works

In [ ]:
!python ml/scripts/sanity_check_dataset.py
!python ml/scripts/sanity_check_transforms.py
!python ml/scripts/sanity_check_head.py

## 4. (Optional) Mount Google Drive for checkpoints
Set `USE_DRIVE = True` to persist run artifacts across Colab sessions.

In [ ]:
USE_DRIVE = False
RUNS_ROOT = '/content/hotdog/ml/runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS_ROOT = '/content/drive/MyDrive/hotdog_ml_runs'
    !mkdir -p {RUNS_ROOT}
print('Runs will be written to:', RUNS_ROOT)

## 5. Train EfficientNet-B0 (main model)

In [ ]:
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
!python ml/scripts/run_training.py \
    --backbone efficientnet_b0 \
    --runs-root {RUNS_ROOT} \
    --run-name {ts}_efficientnet_b0 \
    --batch-size 32 \
    --num-workers 2 \
    --stage1-epochs 4 \
    --stage2-epochs 15

## 6. Train ResNet50 (baseline for comparison)

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
!python ml/scripts/run_training.py \
    --backbone resnet50 \
    --runs-root {RUNS_ROOT} \
    --run-name {ts}_resnet50 \
    --batch-size 32 \
    --num-workers 2 \
    --stage1-epochs 4 \
    --stage2-epochs 15

## 7. Quick look at metrics

In [ ]:
import pandas as pd
from pathlib import Path

for run in sorted(Path(RUNS_ROOT).iterdir()):
    metrics_path = run / 'metrics.csv'
    if not metrics_path.exists():
        continue
    df = pd.read_csv(metrics_path)
    print(f'\n=== {run.name} ===')
    print(df.to_string(index=False))